# 3B · Cleaning the Horror File
### Financial Analytics — Module 3

In Module 1 you audited `messy_transactions.csv` by eye and wrote a Trust Report. Verdict: **not fit for analysis**. Now you fix every defect with code.

This is the most *employable* notebook in Part A — data cleaning is 60–80% of real analyst work, and this file contains every classic defect:
duplicates, three date formats, epoch dates, missing categories, merchant name variants, currency contamination, impossible amounts, and whitespace noise.

In [ ]:
import pandas as pd
import numpy as np

import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
txn = pd.read_csv(BASE + "messy_transactions.csv")
print(txn.shape)
txn.head()

## Step 0 — Measure before you clean

Professional rule: **count the problems before fixing them**, so you can prove what you changed. This is Module 1's auditability pillar in action.

In [ ]:
report = {
    "rows": len(txn),
    "exact_duplicates": txn.duplicated().sum(),
    "missing_category": txn["category"].isna().sum(),
    "epoch_dates": (txn["txn_date"] == "1970-01-01").sum(),
    "zero_amounts": (txn["amount_inr"] == 0).sum(),
    "negative_amounts": (txn["amount_inr"] < 0).sum(),
    "padded_cities": txn["city"].str.strip().ne(txn["city"]).sum(),
}
for k, v in report.items():
    print(f"{k:<20} {v}")

Compare these counts with the dataset registry — they should match exactly. If your numbers ever *don't* match a dataset's documentation, believe neither until you know why.

---
## Defect 1 — Exact duplicates

A double-posting incident. `drop_duplicates()` keeps the first copy of each.

In [ ]:
before = len(txn)
txn = txn.drop_duplicates()
print(f"Removed {before - len(txn)} duplicate rows -> {len(txn)} remain")

---
## Defect 2 — Three date formats (the dangerous one)

The file mixes `2024-07-20`, `04/02/2025` and `Jan 12, 24`. The slash format is **ambiguous**: is `04/02/2025` 4 Feb or 2 Apr?

The registry tells us the source system used day-first (`%d/%m/%Y`). Without that documentation we'd have to go ask — this is why lineage matters. **Never guess a date format.**

In [ ]:
def parse_mixed_date(s):
    """Parse the three known formats. Anything else -> NaT (not-a-time)."""
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%b %d, %y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

txn["txn_date_clean"] = txn["txn_date"].apply(parse_mixed_date)

print("Unparseable:", txn["txn_date_clean"].isna().sum())
txn[["txn_date", "txn_date_clean"]].sample(5, random_state=1)

---
## Defect 3 — Epoch dates (1970-01-01)

Module 1 taught you: `1970-01-01` means *missing timestamp*, not time travel. So convert them to genuinely missing (`NaT`) rather than letting them poison any time analysis.

In [ ]:
epoch_mask = txn["txn_date_clean"] == "1970-01-01"
print("Epoch rows:", epoch_mask.sum())

txn.loc[epoch_mask, "txn_date_clean"] = pd.NaT
print("Now missing dates:", txn["txn_date_clean"].isna().sum())

---
## Defect 4 — Whitespace and case noise in `city`

`" MUMBAI "` and `"Mumbai"` are the same city, but `groupby` doesn't know that.

In [ ]:
print("Distinct city strings BEFORE:", txn["city"].nunique())

txn["city"] = txn["city"].str.strip().str.title()

print("Distinct city strings AFTER: ", txn["city"].nunique())
txn["city"].value_counts()

---
## Defect 5 — Merchant name variants (entity resolution)

Same merchant, five spellings. In real banks this problem has whole teams; at our scale a mapping dict does it.

In [ ]:
txn["merchant"] = txn["merchant"].str.strip()

merchant_map = {
    "BIG BAZAAR": "BigBazaar",
    "IndianOil": "Indian Oil",
    "amazon india": "Amazon India",
    "TATA POWER LTD": "Tata Power",
    "Zerodha Broking": "Zerodha",
}
txn["merchant"] = txn["merchant"].replace(merchant_map)

txn["merchant"].value_counts()

**How did we build that map?** By looking: `txn["merchant"].value_counts()` before cleaning shows the variants sitting next to their parents. Run it yourself on the raw file to see.

---
## Defect 6 — Impossible amounts

Zero and negative amounts can't be real retail transactions. **Decision, not deletion:** we flag and quarantine rather than silently drop — someone upstream should be told.

In [ ]:
bad_amount = (txn["amount_inr"] <= 0)
quarantine = txn[bad_amount].copy()
txn = txn[~bad_amount]                      # ~ means NOT

print(f"Quarantined {len(quarantine)} rows with amount <= 0")
quarantine[["txn_id", "merchant", "amount_inr"]].head()

---
## Defect 7 — Currency contamination (the silent killer)

12 rows are USD amounts, unmarked — they look like tiny INR transactions (₹3 at a fuel pump?). No flag identifies them, so we **detect statistically, verify by eye, then decide.**

In [ ]:
# Suspiciously small amounts for their category
suspects = txn[txn["amount_inr"] < 20].copy()
print(len(suspects), "suspiciously small transactions")
suspects[["merchant", "category", "amount_inr"]].head(12)

In [ ]:
# The registry confirms: unmarked USD at ~83 INR/USD. Convert and flag.
usd_mask = txn["amount_inr"] < 20
txn.loc[usd_mask, "amount_inr"] = (txn.loc[usd_mask, "amount_inr"] * 83.0).round(2)
txn["was_usd"] = usd_mask                       # keep the audit trail!

print("Converted:", usd_mask.sum(), "rows | flagged in column 'was_usd'")

<div style="border-left:4px solid #DC2626;padding:8px 12px">
<b>Honest caveat:</b> in real work you would <i>never</i> convert on a statistical hunch alone — you'd confirm with the source system first. We can convert here only because the registry documents the defect. The habit to keep: <b>detect statistically, confirm with provenance, convert with an audit flag.</b>
</div>

---
## Defect 8 — Missing categories

190 uncategorised transactions. Three honest options: leave as `NaN`, label as `"Uncategorised"`, or infer from merchant. We'll infer where the merchant makes it obvious, and label the rest.

In [ ]:
merchant_to_category = {
    "Swiggy": "Dining", "IRCTC": "Travel", "Apollo Pharmacy": "Healthcare",
    "Indian Oil": "Fuel", "Tata Power": "Utilities", "BSNL": "Utilities",
    "Uber India": "Travel", "Zerodha": "Investment",
}

missing = txn["category"].isna()
txn.loc[missing, "category"] = txn.loc[missing, "merchant"].map(merchant_to_category)
txn["category"] = txn["category"].fillna("Uncategorised")

print("Still labelled 'Uncategorised':", (txn["category"] == "Uncategorised").sum())

---
## The after picture — and the payoff

In [ ]:
print("CLEANING SUMMARY")
print(f"  Started with : 5,060 rows")
print(f"  Duplicates   : -54")
print(f"  Quarantined  : -{len(quarantine)} (impossible amounts)")
print(f"  Final        : {len(txn)} rows")
print(f"  Dates parsed : {txn['txn_date_clean'].notna().sum()} ({txn['txn_date_clean'].isna().sum()} genuinely unknown)")
print(f"  USD converted: {txn['was_usd'].sum()} (flagged)")

# NOW analysis is trustworthy - spending by category:
txn.groupby("category")["amount_inr"].agg(["count", "sum", "mean"]).round(0).sort_values("sum", ascending=False)

That table would have been **wrong in every row** on the raw file — inflated by duplicates, distorted by unconverted USD, fragmented by city and merchant variants. Cleaning isn't preparation for analysis. It **is** analysis.

### ✏️ Exercises
1. Which **city** has the highest total spend after cleaning? Run the same groupby on the *raw* file — how different is the answer?
2. Save the clean file with `txn.to_csv("transactions_clean.csv", index=False)`.
3. Monthly spend: use `txn["txn_date_clean"].dt.to_period("M")` in a groupby. Do the missing-date rows matter for this? What did you do with them, and why?

---
*AI disclosure: ______*